# Unidad 1 · Colab 2 de 3
## Principios de POO y métodos especiales

**Objetivos de este notebook**

- Aplicar los cuatro pilares de la POO: encapsulamiento, herencia, polimorfismo y abstracción.
- Usar `@property` para exponer atributos con validación (getters/setters pythónicos).
- Implementar métodos especiales (`__init__`, `__repr__`, `__str__`, entre otros).

> **Nivel:** intermedio. Este notebook retoma las clases `Producto` y `Carrito` del Colab 1.

---

## 1. Encapsulamiento

Python no tiene atributos verdaderamente privados, pero usa convenciones:

- `_atributo`: protegido por convención (uso interno, pero accesible).
- `__atributo`: *name mangling* (Python lo renombra internamente para dificultar el acceso accidental desde fuera).

```python
class CuentaBancaria:
    def __init__(self, saldo_inicial):
        self._saldo = saldo_inicial  # protegido por convencion

    def depositar(self, monto):
        self._saldo += monto

    def consultar_saldo(self):
        return self._saldo
```

El encapsulamiento protege **invariantes**: reglas que siempre deben cumplirse (por ejemplo, que el saldo nunca sea negativo), controlando cómo se modifica el estado interno en vez de exponerlo directamente.

## 2. `@property`: getters y setters pythónicos

```python
class CuentaBancaria:
    def __init__(self, saldo_inicial):
        self._saldo = saldo_inicial

    @property
    def saldo(self):
        return self._saldo

    @saldo.setter
    def saldo(self, valor):
        if valor < 0:
            raise ValueError('El saldo no puede ser negativo')
        self._saldo = valor

cuenta = CuentaBancaria(100)
print(cuenta.saldo)   # se lee como atributo, aunque es un metodo
cuenta.saldo = 200    # se asigna como atributo, pero pasa por la validacion
```

`@property` deja que el código que usa la clase siga escribiendo `cuenta.saldo` (como si fuera un atributo simple), mientras por dentro corre la validación — sin necesitar `cuenta.get_saldo()` / `cuenta.set_saldo(...)` como en otros lenguajes.

Documentación oficial: [property()](https://docs.python.org/3/library/functions.html#property)

### Ejercicio 1 — Encapsular con `@property`

Modificá la clase `Producto` del Colab 1 para que `precio` sea una `@property` con su setter, que lance un `ValueError` si se intenta asignar un precio menor o igual a cero.

<details>
<summary>💡 Ver solución</summary>

```python
class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio  # usa el setter de abajo

    @property
    def precio(self):
        return self._precio

    @precio.setter
    def precio(self, valor):
        if valor <= 0:
            raise ValueError('El precio debe ser mayor a cero')
        self._precio = valor

p = Producto('Mouse', 15.0)
try:
    p.precio = -5
except ValueError as e:
    print(e)
```

</details>

## 3. Herencia

Una clase puede heredar de otra, reutilizando sus atributos y métodos, y agregando o sobreescribiendo lo que necesite.

```python
class Usuario:
    def __init__(self, nombre, email):
        self.nombre = nombre
        self.email = email

    def descripcion(self):
        return f'{self.nombre} ({self.email})'

class UsuarioPremium(Usuario):
    def __init__(self, nombre, email, descuento):
        super().__init__(nombre, email)  # reutiliza el __init__ del padre
        self.descuento = descuento

    def descripcion(self):  # sobreescribe (override) el metodo del padre
        return f'{super().descripcion()} - Premium ({self.descuento}% off)'

u = UsuarioPremium('Ana', 'ana@mail.com', 15)
print(u.descripcion())
```

### Ejercicio 2 — Jerarquía de clases

Creá una clase base `Empleado` con `nombre` y `salario_base`, y un método `salario_final(self)` que devuelva `salario_base`. Creá una subclase `Vendedor(Empleado)` que además tenga `comision`, y sobreescriba `salario_final` para devolver `salario_base + comision`.

<details>
<summary>💡 Ver solución</summary>

```python
class Empleado:
    def __init__(self, nombre, salario_base):
        self.nombre = nombre
        self.salario_base = salario_base

    def salario_final(self):
        return self.salario_base

class Vendedor(Empleado):
    def __init__(self, nombre, salario_base, comision):
        super().__init__(nombre, salario_base)
        self.comision = comision

    def salario_final(self):
        return self.salario_base + self.comision

e = Empleado('Bruno', 1000)
v = Vendedor('Carla', 1000, 300)
print(e.salario_final(), v.salario_final())
```

</details>

## 4. Polimorfismo

Distintas clases pueden responder al mismo método con comportamientos distintos. Esto permite tratar objetos de clases diferentes de manera uniforme:

```python
empleados = [Empleado('Bruno', 1000), Vendedor('Carla', 1000, 300)]

for e in empleados:
    print(e.nombre, '->', e.salario_final())  # cada uno usa SU version de salario_final
```

Python además tiene *duck typing*: no hace falta una relación de herencia formal para que el polimorfismo funcione — si dos clases distintas tienen un método con el mismo nombre, ambas pueden usarse de forma intercambiable donde se llame a ese método.

### Ejercicio 3 — Polimorfismo con formas

Creá dos clases independientes (sin herencia entre ellas) `Circulo` (con `radio`) y `Rectangulo` (con `base` y `altura`), cada una con un método `area(self)`. Escribí una función `area_total(formas)` que reciba una lista de objetos de cualquiera de las dos clases y sume sus áreas, sin importarle de qué clase es cada uno.

<details>
<summary>💡 Ver solución</summary>

```python
import math

class Circulo:
    def __init__(self, radio):
        self.radio = radio

    def area(self):
        return math.pi * self.radio ** 2

class Rectangulo:
    def __init__(self, base, altura):
        self.base = base
        self.altura = altura

    def area(self):
        return self.base * self.altura

def area_total(formas):
    return sum(f.area() for f in formas)

print(area_total([Circulo(2), Rectangulo(3, 4)]))
```

</details>

## 5. Abstracción: clases abstractas

Una clase abstracta define un contrato (qué métodos debe tener una subclase) sin implementarlo del todo — obliga a las subclases a completarlo, y no se puede instanciar directamente.

```python
from abc import ABC, abstractmethod

class MedioDePago(ABC):
    @abstractmethod
    def procesar(self, monto):
        ...

class PagoTarjeta(MedioDePago):
    def procesar(self, monto):
        return f'Cobrando {monto} con tarjeta'

# MedioDePago()  # error: no se puede instanciar una clase abstracta
print(PagoTarjeta().procesar(100))
```

Documentación oficial: [módulo abc](https://docs.python.org/3/library/abc.html)

### Ejercicio 4 — Clase abstracta

Creá una clase abstracta `Notificador(ABC)` con un método abstracto `enviar(self, mensaje)`. Creá dos subclases concretas: `NotificadorEmail` y `NotificadorSMS`, cada una implementando `enviar` de forma distinta (por ejemplo, con un `print` que simule el envío).

<details>
<summary>💡 Ver solución</summary>

```python
from abc import ABC, abstractmethod

class Notificador(ABC):
    @abstractmethod
    def enviar(self, mensaje):
        ...

class NotificadorEmail(Notificador):
    def enviar(self, mensaje):
        print(f'Enviando email: {mensaje}')

class NotificadorSMS(Notificador):
    def enviar(self, mensaje):
        print(f'Enviando SMS: {mensaje}')

for n in [NotificadorEmail(), NotificadorSMS()]:
    n.enviar('Tu pedido fue confirmado')
```

</details>

## 6. Métodos especiales (dunder methods)

| Método | Para qué sirve |
|---|---|
| `__init__` | Constructor: inicializa la instancia |
| `__repr__` | Representación para desarrolladores (debug, consola) — idealmente algo que se pueda copiar y ejecutar |
| `__str__` | Representación para humanos (lo que muestra `print(objeto)`) |
| `__eq__` | Define cuándo dos objetos son iguales (`==`) |
| `__len__` | Permite usar `len(objeto)` |

```python
class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

    def __repr__(self):
        return f'Producto(nombre={self.nombre!r}, precio={self.precio})'

    def __str__(self):
        return f'{self.nombre} - ${self.precio}'

p = Producto('Mouse', 15.0)
print(p)        # usa __str__: Mouse - $15.0
print(repr(p))  # usa __repr__: Producto(nombre='Mouse', precio=15.0)
```

Si no definís `__str__`, Python usa `__repr__` como respaldo — por eso conviene definir al menos `__repr__` siempre.

Documentación oficial: [Nombres de métodos especiales](https://docs.python.org/3/reference/datamodel.html#special-method-names)

### Ejercicio 5 — `__repr__`, `__str__` y `__eq__`

A la clase `Cliente` (con `nombre` y `email`), agregále `__repr__` (formato `Cliente(nombre=..., email=...)`), `__str__` (formato `nombre <email>`), y `__eq__` (dos clientes son iguales si tienen el mismo `email`).

<details>
<summary>💡 Ver solución</summary>

```python
class Cliente:
    def __init__(self, nombre, email):
        self.nombre = nombre
        self.email = email

    def __repr__(self):
        return f'Cliente(nombre={self.nombre!r}, email={self.email!r})'

    def __str__(self):
        return f'{self.nombre} <{self.email}>'

    def __eq__(self, otro):
        return isinstance(otro, Cliente) and self.email == otro.email

c1 = Cliente('Ana', 'ana@mail.com')
c2 = Cliente('Ana Perez', 'ana@mail.com')
print(c1)
print(repr(c1))
print(c1 == c2)  # True, mismo email
```

</details>

## Mini-proyecto: cuenta bancaria completa

Diseñá una jerarquía de clases `Cuenta` con:

1. `saldo` como `@property`, con setter que impida un saldo negativo.
2. Una subclase `CuentaAhorro(Cuenta)` que agregue un método `aplicar_interes(self, tasa)`.
3. `__repr__` y `__str__` bien definidos.
4. Un método abstracto en una clase base `Cuenta(ABC)` llamado `tipo(self)` que cada subclase implemente (por ejemplo, devolviendo `'Ahorro'` o `'Corriente'`).

**Entregable:** la jerarquía completa de clases + al menos 2 instancias de distintas subclases mostrando polimorfismo (por ejemplo, iterando una lista de cuentas e imprimiendo `cuenta.tipo()` de cada una).

---

**Seguís en:** *Colab 3 — Patrones de diseño y modelado de un negocio digital*